# 00 — Snapshot dos discursos em plenário

Filtra as três arenas, preserva os textos, audita duplicações e realiza a junção temporal.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/falando_nela/data")
REPO_DIR = Path("/content/falando_nela")
REPO_URL = "https://github.com/pedblan/falando_nela.git"
REPO_REF = ""  # Opcional: branch, tag ou commit; vazio acompanha o default remoto.

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"], check=True)
    if not REPO_REF:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
if REPO_REF:
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)

os.chdir(REPO_DIR)
os.environ["FALANDO_NELA_DATA_ROOT"] = str(DATA_ROOT)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--force-reinstall",
        "--no-cache-dir",
        "numpy==2.0.2",
        "pandas==2.2.3",
    ],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-analise.txt"], check=True)
ABI_CHECK = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import numpy as np; import pandas as pd; "
            "assert np.__version__ == '2.0.2', np.__version__; "
            "assert pd.__version__ == '2.2.3', pd.__version__; "
            "print(f'NumPy {np.__version__}; pandas {pd.__version__}')"
        ),
    ],
    check=True,
    text=True,
    capture_output=True,
)
import numpy as np
import pandas as pd

assert np.__version__ == "2.0.2", f"Reinicie a sessao do Colab: NumPy carregado={np.__version__}"
assert pd.__version__ == "2.2.3", f"Reinicie a sessao do Colab: pandas carregado={pd.__version__}"
print("Data root:", DATA_ROOT)
print("Commit:", subprocess.run(["git", "rev-parse", "HEAD"], check=True, text=True, capture_output=True).stdout.strip())
print("ABI:", ABI_CHECK.stdout.strip())

## Configuração

Use o mesmo `RUN_ID` em toda a suíte. A configuração versionada é a fonte de verdade.

In [ ]:
from analise.discursos_plenario.config import load_config, resolve_input_paths, resolve_output_root

RUN_ID = "analise-plenario-20260713-v1"
CONFIG_PATH = REPO_DIR / "analise" / "discursos_plenario" / "config.v1.json"
ANALYSIS_CONFIG = load_config(CONFIG_PATH)
RUN_OUTPUT_ROOT = resolve_output_root(ANALYSIS_CONFIG, DATA_ROOT, RUN_ID)
INPUT_PATHS = resolve_input_paths(ANALYSIS_CONFIG, DATA_ROOT)
RODAR_ETAPA = False

assert ANALYSIS_CONFIG.date_start == "2010-02-02"
assert ANALYSIS_CONFIG.date_end == "2026-07-13"
assert ANALYSIS_CONFIG.raw["complete_year_end"] == 2025
assert ANALYSIS_CONFIG.raw["ytd_year"] == 2026
print("Run:", RUN_ID)
print("Saida:", RUN_OUTPUT_ROOT)

## Decisão metodológica

Revise primeiro o inventário de entradas. A etapa é imutável por `RUN_ID`: se dados ou configuração mudarem, crie outro run.

In [ ]:
import pandas as pd
from analise.discursos_plenario.io import input_inventory

SNAPSHOT_INVENTORY = input_inventory(ANALYSIS_CONFIG, DATA_ROOT)
display(SNAPSHOT_INVENTORY)
SNAPSHOT_REQUIRED = ["camara", "senado", "congresso", "parliamentarian_periods"]
SNAPSHOT_MISSING = SNAPSHOT_INVENTORY.loc[
    SNAPSHOT_INVENTORY["entrada"].isin(SNAPSHOT_REQUIRED) & ~SNAPSHOT_INVENTORY["existe"], "caminho"
].tolist()
assert not SNAPSHOT_MISSING, f"Entradas obrigatorias ausentes: {SNAPSHOT_MISSING}"

## Execução

A etapa cara permanece desativada até a inspeção das entradas e dos parâmetros acima.

In [ ]:
from analise.discursos_plenario.snapshot import run_snapshot

SNAPSHOT_RESULT = None
if RODAR_ETAPA:
    SNAPSHOT_RESULT = run_snapshot(
        data_root=DATA_ROOT,
        run_id=RUN_ID,
        config_path=CONFIG_PATH,
        overwrite=False,
    )
    print(SNAPSHOT_RESULT["manifest_path"])
else:
    print("Etapa não executada. Revise o inventário e defina RODAR_ETAPA=True.")

## Validação imediata

Esta checagem não substitui os testes sintéticos nem a revisão dos manifests.

In [ ]:
from pathlib import Path

import pandas as pd


SNAPSHOT_RUN_ID = globals().get("RUN_ID", "analise-plenario-20260713-v1")
SNAPSHOT_DATA_ROOT = Path(
    globals().get("DATA_ROOT", "/content/drive/MyDrive/falando_nela/data")
)
SNAPSHOT_RUN_OUTPUT_ROOT = Path(
    globals().get(
        "RUN_OUTPUT_ROOT",
        SNAPSHOT_DATA_ROOT
        / "analises"
        / "discursos_plenario"
        / "v1"
        / SNAPSHOT_RUN_ID,
    )
)
SNAPSHOT_PATH = (
    SNAPSHOT_RUN_OUTPUT_ROOT
    / "00_snapshot"
    / "discursos_plenario_snapshot.parquet"
)

assert SNAPSHOT_PATH.exists(), (
    f"Snapshot não encontrado: {SNAPSHOT_PATH}. "
    "Monte o Drive e confira RUN_ID/DATA_ROOT."
)

SNAPSHOT_FRAME = pd.read_parquet(SNAPSHOT_PATH)
SNAPSHOT_REQUIRED_COLUMNS = {
    "arena",
    "ano",
    "data_analise",
    "elegivel_inferencia_anual",
}
SNAPSHOT_MISSING_COLUMNS = SNAPSHOT_REQUIRED_COLUMNS.difference(SNAPSHOT_FRAME.columns)
assert not SNAPSHOT_MISSING_COLUMNS, (
    f"Colunas obrigatórias ausentes: {sorted(SNAPSHOT_MISSING_COLUMNS)}"
)

SNAPSHOT_EXPECTED_ARENAS = ["camara", "senado", "congresso"]
SNAPSHOT_OBSERVED_ARENAS = set(SNAPSHOT_FRAME["arena"].dropna().astype(str).unique())
assert SNAPSHOT_OBSERVED_ARENAS == set(SNAPSHOT_EXPECTED_ARENAS), (
    f"Arenas observadas: {sorted(SNAPSHOT_OBSERVED_ARENAS)}; "
    f"esperadas: {sorted(SNAPSHOT_EXPECTED_ARENAS)}"
)

SNAPSHOT_FRAME["data_analise"] = pd.to_datetime(
    SNAPSHOT_FRAME["data_analise"], errors="coerce"
)
SNAPSHOT_FRAME["ano"] = pd.to_numeric(SNAPSHOT_FRAME["ano"], errors="coerce")
assert SNAPSHOT_FRAME["data_analise"].notna().all(), "Há data_analise inválida."
assert SNAPSHOT_FRAME["ano"].notna().all(), "Há ano inválido."
SNAPSHOT_FRAME["ano"] = SNAPSHOT_FRAME["ano"].astype(int)

SNAPSHOT_DATE_START = pd.Timestamp("2010-02-02")
SNAPSHOT_DATE_END = pd.Timestamp("2026-07-13")
assert SNAPSHOT_FRAME["data_analise"].between(
    SNAPSHOT_DATE_START, SNAPSHOT_DATE_END, inclusive="both"
).all(), "Há discursos fora do recorte temporal."
assert not SNAPSHOT_FRAME.loc[
    SNAPSHOT_FRAME["ano"].eq(2026), "elegivel_inferencia_anual"
].fillna(False).any(), "2026 não pode ser elegível à inferência anual."

SNAPSHOT_SUMMARY = (
    SNAPSHOT_FRAME.groupby("arena", observed=True)
    .agg(
        discursos=("arena", "size"),
        ano_inicial=("ano", "min"),
        ano_final=("ano", "max"),
        anos_com_discursos=("ano", "nunique"),
        data_inicial=("data_analise", "min"),
        data_final=("data_analise", "max"),
    )
    .reindex(SNAPSHOT_EXPECTED_ARENAS)
)

SNAPSHOT_COUNTS = (
    SNAPSHOT_FRAME.groupby(["ano", "arena"], observed=True)
    .size()
    .rename("discursos")
    .unstack("arena")
)
SNAPSHOT_COVERAGE = (
    SNAPSHOT_COUNTS.reindex(
        index=range(2010, 2027),
        columns=SNAPSHOT_EXPECTED_ARENAS,
        fill_value=0,
    )
    .fillna(0)
    .astype("int64")
)
SNAPSHOT_COVERAGE.index.name = "ano"
SNAPSHOT_COVERAGE.columns.name = "arena"

display(SNAPSHOT_SUMMARY)
display(SNAPSHOT_COVERAGE)

SNAPSHOT_MISSING_YEARS = {
    arena: SNAPSHOT_COVERAGE.index[SNAPSHOT_COVERAGE[arena].eq(0)].tolist()
    for arena in SNAPSHOT_EXPECTED_ARENAS
}
print("Anos sem discursos no snapshot:")
for SNAPSHOT_ARENA in SNAPSHOT_EXPECTED_ARENAS:
    print(f"- {SNAPSHOT_ARENA}: {SNAPSHOT_MISSING_YEARS[SNAPSHOT_ARENA] or 'nenhum'}")

print("Snapshot validado:", SNAPSHOT_PATH)
print("Total de discursos:", len(SNAPSHOT_FRAME))